# Feature Engineering

Purpose:<br>
Create predictors available at or before quarter t for predicting whether a
MySuper investment option will fall into the bottom quartile of peer performance
over quarters t+1 to t+4.

Target:<br>
future_4q_bottom_quartile

Rule:<br>
No feature may use information after period_end_date.

## Data Import from audit file

In [1]:
import pandas as pd
from pathlib import Path

processed_dir = Path("../data/processed")

hist_model_mysuper_features = pd.read_parquet(
    processed_dir / "hist_model_mysuper_target_peer.parquet"
)

In [2]:
hist_model_mysuper_features.shape

(13719, 110)

In [3]:
hist_model_mysuper_features[
    "future_4q_target_available_option"
].value_counts(dropna=False)

future_4q_target_available_option
True     11720
False     1999
Name: count, dtype: int64

In [4]:
hist_model_mysuper_features[
    [
        "period_end_date",
        "target_end_date",
        "future_4q_bottom_quartile",
    ]
].dtypes

period_end_date              datetime64[us]
target_end_date              datetime64[us]
future_4q_bottom_quartile           boolean
dtype: object

In [5]:
# define entity keys
entity_keys = [
    "rse_abn",
    "abn_product_identifier",
    "abn_investment_menu_identifier",
    "abn_investment_option_identifier",
]

# sort values based on entity keys and quarter
hist_model_mysuper_features = (
    hist_model_mysuper_features
    .sort_values(
        entity_keys + ["period_end_date"]
    )
    .reset_index(drop=True)
)

## Setting performance feature set

In [6]:
return_col = "return_measurement_comparison_percent"

# create lagged return columns
grouped_returns = (
    hist_model_mysuper_features
    .groupby(entity_keys)[return_col]
)

hist_model_mysuper_features["return_lag_1q"] = (
    grouped_returns.shift(1)
)

hist_model_mysuper_features["return_lag_2q"] = (
    grouped_returns.shift(2)
)

hist_model_mysuper_features["return_lag_4q"] = (
    grouped_returns.shift(4)
)

interpretation:<br>
- return_measurement_comparison_percent = return at t
- return_lag_1q                         = return at t-1
- return_lag_2q                         = return at t-2
- return_lag_4q                         = return at t-4

In [7]:
# add deterioration feature
# using an early-warning concept to know whether performance has weakened relative to the previous quarter
hist_model_mysuper_features[
    "return_change_1q"
] = (
    hist_model_mysuper_features[
        return_col
    ]
    -
    hist_model_mysuper_features[
        "return_lag_1q"
    ]
)

In [8]:
# create quater identifiers and lagged quarter dates
# validate whether the lag created before really 1,2,4 quarter ago
hist_model_mysuper_features[
    "quarter"
] = (
    hist_model_mysuper_features[
        "period_end_date"
    ].dt.to_period("Q")
)

grouped_quarters = (
    hist_model_mysuper_features
    .groupby(entity_keys)["quarter"]
)

hist_model_mysuper_features[
    "quarter_lag_1q"
] = grouped_quarters.shift(1)

hist_model_mysuper_features[
    "quarter_lag_2q"
] = grouped_quarters.shift(2)

hist_model_mysuper_features[
    "quarter_lag_4q"
] = grouped_quarters.shift(4)

# Check whether each lag is genuinely the requested quarter
hist_model_mysuper_features[
    "return_lag_1q_valid"
] = (
    hist_model_mysuper_features["quarter_lag_1q"]
    ==
    hist_model_mysuper_features["quarter"] - 1
)

hist_model_mysuper_features[
    "return_lag_2q_valid"
] = (
    hist_model_mysuper_features["quarter_lag_2q"]
    ==
    hist_model_mysuper_features["quarter"] - 2
)

hist_model_mysuper_features[
    "return_lag_4q_valid"
] = (
    hist_model_mysuper_features["quarter_lag_4q"]
    ==
    hist_model_mysuper_features["quarter"] - 4
)

In [9]:
# inspect result
hist_model_mysuper_features[
    [
        "return_lag_1q_valid",
        "return_lag_2q_valid",
        "return_lag_4q_valid",
    ]
].apply(
    lambda x: x.value_counts(dropna=False)
)

,return_lag_1q_valid,return_lag_2q_valid,return_lag_4q_valid
True,13204,12689,11674
False,515,1030,2045


this pattern looks largely like the beginning of each investment option's history. For example, the first observation of an option cannot have a 1-quarter lag; the first two cannot have a 2-quarter lag; the first four cannot have a 4-quarter lag.

But False might have 2 meanings:
- No earlier observation exists → expected insufficient history
- An earlier observation exists, but it is not the required calendar quarter → genuine gap

The second meaning has to be handled

In [10]:
# check actual gap
for lag in [1, 2, 4]:
    valid_col = f"return_lag_{lag}q_valid"
    quarter_lag_col = f"quarter_lag_{lag}q"

    actual_gap = (
        ~hist_model_mysuper_features[valid_col]
        &
        hist_model_mysuper_features[quarter_lag_col].notna()
    )

    no_prior_history = (
        ~hist_model_mysuper_features[valid_col]
        &
        hist_model_mysuper_features[quarter_lag_col].isna()
    )

    print(f"{lag}Q lag")
    print("Valid:", hist_model_mysuper_features[valid_col].sum())
    print("No prior history:", no_prior_history.sum())
    print("Actual gap:", actual_gap.sum())
    print()

1Q lag
Valid: 13204
No prior history: 515
Actual gap: 0

2Q lag
Valid: 12689
No prior history: 1030
Actual gap: 0

4Q lag
Valid: 11674
No prior history: 2045
Actual gap: 0



In [11]:
import numpy as np

# mask the invalid lag values
for lag in [1, 2, 4]:
    hist_model_mysuper_features.loc[
        ~hist_model_mysuper_features[
            f"return_lag_{lag}q_valid"
        ],
        f"return_lag_{lag}q",
    ] = np.nan

# recalculate deterioration
hist_model_mysuper_features[
    "return_change_1q"
] = (
    hist_model_mysuper_features[
        "return_measurement_comparison_percent"
    ]
    -
    hist_model_mysuper_features[
        "return_lag_1q"
    ]
)

#### Create trailing 4-quarter cumulative return

In [12]:
# create 3-quarter lag
hist_model_mysuper_features[
    "return_lag_3q"
] = (
    hist_model_mysuper_features
    .groupby(entity_keys)[
        "return_measurement_comparison_percent"
    ]
    .shift(3)
)

# Create trailing cumulative return
hist_model_mysuper_features[
    "return_trailing_4q"
] = (
    (1 + hist_model_mysuper_features[
        "return_measurement_comparison_percent"
    ])
    *
    (1 + hist_model_mysuper_features[
        "return_lag_1q"
    ])
    *
    (1 + hist_model_mysuper_features[
        "return_lag_2q"
    ])
    *
    (1 + hist_model_mysuper_features[
        "return_lag_3q"
    ])
    - 1
)

In [13]:
# check distribution
hist_model_mysuper_features[
    "return_trailing_4q"
].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

count    12176.000000
mean         0.075215
std          0.071414
min         -0.114451
1%          -0.084385
5%          -0.058948
25%          0.036310
50%          0.082569
75%          0.115614
95%          0.191143
99%          0.255577
max          0.372972
Name: return_trailing_4q, dtype: float64

In [14]:
hist_model_mysuper_features[
    "return_trailing_4q"
].isna().value_counts()

return_trailing_4q
False    12176
True      1543
Name: count, dtype: int64

In [15]:
return_col = "return_measurement_comparison_percent"

# group the table based on previous entity keys
grouped_returns = (
    hist_model_mysuper_features
    .groupby(entity_keys)[return_col]
)

# calculate mean over 4 quarters
hist_model_mysuper_features[
    "return_mean_4q"
] = (
    grouped_returns
    .rolling(window=4, min_periods=4)
    .mean()
    .reset_index(level=entity_keys, drop=True)
)

# calculate standard deviation over 4 quarters
hist_model_mysuper_features[
    "return_std_4q"
] = (
    grouped_returns
    .rolling(window=4, min_periods=4)
    .std()
    .reset_index(level=entity_keys, drop=True)
)

# calculate minimum/worst return during 4 quarters
hist_model_mysuper_features[
    "return_min_4q"
] = (
    grouped_returns
    .rolling(window=4, min_periods=4)
    .min()
    .reset_index(level=entity_keys, drop=True)
)

# calculate positive proportion of the return over 4 quarters
hist_model_mysuper_features[
    "positive_quarter_share_4q"
] = (
    grouped_returns
    .rolling(window=4, min_periods=4)
    .apply(lambda x: (x > 0).mean(), raw=True)
    .reset_index(level=entity_keys, drop=True)
)

<b>Meanings:</b><br>
return_trailing_4q = total compounded return over t-3 to t

return_mean_4q = average quarterly return over t-3 to t

return_std_4q = variability of quarterly returns over t-3 to t

return_min_4q = worst quarterly return over t-3 to t

positive_quarter_share_4q = proportion of the last 4 quarters with positive returns

In [16]:
# inspect results
performance_features_4q = [
    "return_trailing_4q",
    "return_mean_4q",
    "return_std_4q",
    "return_min_4q",
    "positive_quarter_share_4q",
]

hist_model_mysuper_features[
    performance_features_4q
].describe()

,return_trailing_4q,return_mean_4q,return_std_4q,return_min_4q,positive_quarter_share_4q
count,12176.000000,12176.000000,12176.000000,12176.000000,12176.000000
mean,0.075215,0.018438,0.032825,-0.023338,0.755626
std,0.071414,0.016664,0.020246,0.039705,0.195507
min,-0.114451,-0.028650,0.000000,-0.177122,0.000000
25%,0.036310,0.010156,0.019993,-0.040755,0.750000
50%,0.082569,0.020413,0.027996,-0.010104,0.750000
75%,0.115614,0.027950,0.038938,0.001102,1.000000
max,0.372972,0.082475,0.131923,0.078200,1.000000


In [17]:
hist_model_mysuper_features[
    performance_features_4q
].isna().sum()

return_trailing_4q           1543
return_mean_4q               1543
return_std_4q                1543
return_min_4q                1543
positive_quarter_share_4q    1543
dtype: int64

All features have the same 1,543 missing values because all of them use the same four-quarter window. Those are mainly early observations that do not yet have four quarters of historical data.

Next, create features that compare target in similar characteristics in the same quarter (compare in peer group).

#### Peer-relative historical performance

Since an investment option can appear multiple times through different product/menu representations, if median or percentile directly calculated from the table, then some investment options would receive multiple "votes" again. Because of that, the same principle  established on the current return field has to be established: one investment option per quarter when constructing the peer benchmark.

In [18]:
# check whether the current-quarter return is consistent across repeated representations of an option-quarter
peer_keys = [
    "rse_abn",
    "abn_investment_option_identifier",
    "period_end_date",
]

current_return_consistency = (
    hist_model_mysuper_features
    .groupby(peer_keys)[
        "return_measurement_comparison_percent"
    ]
    .nunique(dropna=False)
)

current_return_consistency.value_counts().sort_index()

return_measurement_comparison_percent
1    11637
2       14
Name: count, dtype: int64

almost every investment option-quarter has one unique current return, but 14 option-quarters have two different current returns across their product/menu representations.

In [19]:
return_col = "return_measurement_comparison_percent"

current_return_conflicts = (
    hist_model_mysuper_features
    .groupby(peer_keys)[return_col]
    .agg(
        distinct_returns=lambda x: x.nunique(dropna=False),
        min_return="min",
        max_return="max",
    )
    .reset_index()
)

current_return_conflicts = (
    current_return_conflicts.loc[
        current_return_conflicts[
            "distinct_returns"
        ] > 1
    ]
    .copy()
)

current_return_conflicts[
    "return_difference"
] = (
    current_return_conflicts["max_return"]
    -
    current_return_conflicts["min_return"]
)

current_return_conflicts

,rse_abn,abn_investment_option_identifier,period_end_date,distinct_returns,min_return,max_return,return_difference
1147,19905422981,19905422981-SP194953,2022-06-30,2,-0.049373,-0.031721,0.017652
1193,19905422981,19905422981-SP195458,2022-06-30,2,-0.050285,-0.032545,0.017740
1239,19905422981,19905422981-SP195963,2022-06-30,2,-0.061160,-0.039881,0.021279
1285,19905422981,19905422981-SP196468,2022-06-30,2,-0.069831,-0.045669,0.024162
1331,19905422981,19905422981-SP196973,2022-06-30,2,-0.075980,-0.050160,0.025820
1377,19905422981,19905422981-SP197478,2022-06-30,2,-0.075671,-0.049865,0.025806
1423,19905422981,19905422981-SP197983,2022-06-30,2,-0.075963,-0.049964,0.025999
1469,19905422981,19905422981-SP198488,2022-06-30,2,-0.076181,-0.050198,0.025983
1515,19905422981,19905422981-SP198993,2022-06-30,2,-0.077522,-0.051068,0.026454
1561,19905422981,19905422981-SP199498,2022-06-30,2,-0.077175,-0.050720,0.026455


In [20]:
conflicting_current_return_rows = (
    hist_model_mysuper_features
    .merge(
        current_return_conflicts[peer_keys],
        on=peer_keys,
        how="inner",
    )
    .sort_values(
        peer_keys
        + [
            "abn_product_identifier",
            "abn_investment_menu_identifier",
        ]
    )
)

conflicting_current_return_rows[
    [
        "rse_abn",
        "abn_product_identifier",
        "abn_investment_menu_identifier",
        "abn_investment_option_identifier",
        "period_end_date",
        "return_measurement_comparison_percent",
    ]
]

,rse_abn,abn_product_identifier,abn_investment_menu_identifier,abn_investment_option_identifier,period_end_date,return_measurement_comparison_percent
0,19905422981,19905422981-CSDMYSUPER1,19905422981-CSDMYSUPER1,19905422981-SP194953,2022-06-30,-0.049373
12,19905422981,19905422981-CSDMYSUPER2,19905422981-CSDMYSUPER1,19905422981-SP194953,2022-06-30,-0.031721
24,19905422981,19905422981-CSDMYSUPER3,19905422981-CSDMYSUPER1,19905422981-SP194953,2022-06-30,-0.049373
36,19905422981,19905422981-CSDMYSUPER4,19905422981-CSDMYSUPER1,19905422981-SP194953,2022-06-30,-0.049373
1,19905422981,19905422981-CSDMYSUPER1,19905422981-CSDMYSUPER1,19905422981-SP195458,2022-06-30,-0.050285
13,19905422981,19905422981-CSDMYSUPER2,19905422981-CSDMYSUPER1,19905422981-SP195458,2022-06-30,-0.032545
25,19905422981,19905422981-CSDMYSUPER3,19905422981-CSDMYSUPER1,19905422981-SP195458,2022-06-30,-0.050285
37,19905422981,19905422981-CSDMYSUPER4,19905422981-CSDMYSUPER1,19905422981-SP195458,2022-06-30,-0.050285
2,19905422981,19905422981-CSDMYSUPER1,19905422981-CSDMYSUPER1,19905422981-SP195963,2022-06-30,-0.061160
14,19905422981,19905422981-CSDMYSUPER2,19905422981-CSDMYSUPER1,19905422981-SP195963,2022-06-30,-0.039881


In [21]:
current_return_conflicts[
    "return_difference"
].describe()

count    14.000000
mean      0.025983
std       0.010159
min       0.015000
25%       0.022000
50%       0.025902
75%       0.026409
max       0.058500
Name: return_difference, dtype: float64

Based on the consistency result, the conflicts are rare: only 14 option-quarters out of roughly 11.6k. Not only that, 12 of the 14 are concentrated in the same RSE and quarter, and in those cases three product representations agree while one differs. The remaining two seems like isolated two-value conflicts.

Because of that, it might not be too disruptive to do simple aggregation across peers. e simplest choice is the median across its product/menu representations to prevent repeated representations from giving one option multiple votes, while avoiding an arbitrary .drop_duplicates() choice. In the 12 concentrated conflicts, the median will also stay close to the majority-reported value.

In [22]:
# create unique option-quarter table
peer_current_returns = (
    hist_model_mysuper_features
    .groupby(peer_keys, as_index=False)
    .agg(current_return_peer_value=("return_measurement_comparison_percent","median"))
)

# calculate the same-quarter MySuper benchmark
peer_current_returns["current_return_peer_median"] = (
    peer_current_returns
    .groupby("period_end_date")["current_return_peer_value"]
    .transform("median")
)


# create the relative difference performance feature
peer_current_returns["current_return_minus_peer_median"] = (
    peer_current_returns["current_return_peer_value"] - peer_current_returns["current_return_peer_median"]
)

# create the peer percentile feature
peer_current_returns["current_return_peer_percentile"] = (
    peer_current_returns
    .groupby("period_end_date")["current_return_peer_value"]
    .rank(pct=True,method="average")
)

current_return_minus_peer_median defined as the difference between current return and return median of its peers in that quarter. <br>
interpretation:
- $>$ 0  → option performed better than the quarter's typical MySuper peer
- $=$ 0  → around the peer median
- $<$ 0  → option performed worse than the quarter's typical MySuper peer

current_return_peer_percentile defined as the percentile of the current return among other returns of its peers in that quarter. <br>
interpretation:
- 0.10 → around the bottom 10% currently
- 0.50 → around the middle
- 0.90 → around the top 10%

next is mapping this to the modelling table.

In [23]:
# define the peer feature
peer_feature_cols = [
    "rse_abn",
    "abn_investment_option_identifier",
    "period_end_date",
    "current_return_minus_peer_median",
    "current_return_peer_percentile",
]

# map the peer features into the modelling table
hist_model_mysuper_features = (
    hist_model_mysuper_features
    .merge(
        peer_current_returns[peer_feature_cols],
        on=peer_keys,
        how="left",
        validate="many_to_one",
    )
)

In [24]:
# validate result
hist_model_mysuper_features[
    [
        "current_return_minus_peer_median",
        "current_return_peer_percentile",
    ]
].describe()

,current_return_minus_peer_median,current_return_peer_percentile
count,13719.000000,13719.000000
mean,0.000038,0.503432
std,0.013281,0.286459
min,-0.080935,0.003030
25%,-0.005787,0.253111
50%,0.000039,0.508251
75%,0.005820,0.749020
max,0.160587,1.000000


In [25]:
hist_model_mysuper_features[
    [
        "current_return_minus_peer_median",
        "current_return_peer_percentile",
    ]
].isna().sum()

current_return_minus_peer_median    0
current_return_peer_percentile      0
dtype: int64

#### Volatility features

In [26]:
# define the volatility features
volatility_features = [
    "return_investment_five_year_volatility_comparison_percent",
    "return_investment_ten_year_volatility_comparison_percent_clean",
    "volatility_5y_missing",
    "volatility_10y_missing",
]

# inspect null values
hist_model_mysuper_features[
    volatility_features
].isna().sum()

return_investment_five_year_volatility_comparison_percent         6915
return_investment_ten_year_volatility_comparison_percent_clean    9846
volatility_5y_missing                                                0
volatility_10y_missing                                               0
dtype: int64

In [27]:
hist_model_mysuper_features[
    [
        "return_investment_five_year_volatility_comparison_percent",
        "return_investment_ten_year_volatility_comparison_percent_clean",
    ]
].describe(
    percentiles=[
        0.01, 0.05, 0.25,
        0.50, 0.75, 0.95, 0.99
    ]
)

,return_investment_five_year_volatility_comparison_percent,return_investment_ten_year_volatility_comparison_percent_clean
count,6804.000000,3873.000000
mean,0.064389,0.055836
std,0.022944,0.022541
min,0.000000,0.000000
1%,0.000000,0.000000
5%,0.031611,0.000000
25%,0.048736,0.046665
50%,0.062222,0.056912
75%,0.080536,0.069786
95%,0.103800,0.088200


The results aligned with the auditing results before. Possible feature derived from this: difference between 5Y and 10Y volatility to signify whether recent years is more or less stable than longer runs.

In [28]:
hist_model_mysuper_features["volatility_5y_minus_10y"] = (
    hist_model_mysuper_features[
        "return_investment_five_year_volatility_comparison_percent"
    ]
    -
    hist_model_mysuper_features[
        "return_investment_ten_year_volatility_comparison_percent_clean"
    ]
)

In [29]:
hist_model_mysuper_features[
    "volatility_5y_minus_10y"
].isna().sum()

np.int64(9852)

This feature can be interpreted as follows:
- $>$ 0 = 5-year volatility is higher than 10-year volatility → more recent years have been more volatile than the longer-run history

- $<$ 0 = 5-year volatility is lower than 10-year volatility → more recent years have been less volatile

- $≈$ 0 = similar medium- and long-term volatility

#### Investment Strategy / SAA

In [30]:
# define SAA features
saa_strategy_features = [
    "strategic_growth_allocation",
    "growth_allocation_missing",
    "growth_allocation_low_coverage",
    "growth_listing_assumption_used",
    "listing_assumption_share",
    "strategic_growth_above_one",
]

# inspect null values
hist_model_mysuper_features[
    saa_strategy_features
].isna().sum()

strategic_growth_allocation       3829
growth_allocation_missing         3829
growth_allocation_low_coverage    3829
growth_listing_assumption_used    3829
listing_assumption_share          3829
strategic_growth_above_one        3829
dtype: int64

In [31]:
hist_model_mysuper_features[
    [
        "strategic_growth_allocation",
        "listing_assumption_share",
    ]
].describe(
    percentiles=[
        0.01, 0.05, 0.25,
        0.50, 0.75, 0.95, 0.99
    ]
)

,strategic_growth_allocation,listing_assumption_share
count,9890.000000,9890.000000
mean,0.750149,0.002048
std,0.151243,0.014179
min,0.172625,0.000000
1%,0.369801,0.000000
5%,0.481250,0.000000
25%,0.628297,0.000000
50%,0.770000,0.000000
75%,0.887550,0.000000
95%,0.941250,0.000000


There are 3,829 missing values in the quality flags themselves. That means those 3,829 modelling rows do not have the SAA-derived record available at all. So fields like growth_allocation_missing cannot tell anything for those rows because the flag itself came from the SAA feature table. Because of that, create explicit missingness feature directly from that value.

In [32]:
hist_model_mysuper_features[
    "strategic_growth_missing"
] = (
    hist_model_mysuper_features[
        "strategic_growth_allocation"
    ].isna()
)

Interpretation: <br>
- False → strategic growth is available
- True  → strategic growth is unavailable

In [33]:
hist_model_mysuper_features[
    "strategic_growth_missing"
].value_counts(dropna=False)

strategic_growth_missing
False    9890
True     3829
Name: count, dtype: int64

Check broad SAA allocations:

In [34]:
# define the SAA allocation features
broad_saa_features = [
    "allocation_equity",
    "allocation_property",
    "allocation_fixed_income",
    "allocation_infrastructure",
    "allocation_cash",
    "allocation_alternatives",
    "allocation_credit",
]

# inspect missingness
hist_model_mysuper_features[
    broad_saa_features
].isna().sum()

allocation_equity            3829
allocation_property          3829
allocation_fixed_income      3829
allocation_infrastructure    3829
allocation_cash              3829
allocation_alternatives      3829
allocation_credit            3829
dtype: int64

In [35]:
# inspect distribution
hist_model_mysuper_features[
    broad_saa_features
].describe(
    percentiles=[
        0.01, 0.05, 0.25,
        0.50, 0.75, 0.95, 0.99
    ]
)

,allocation_equity,allocation_property,allocation_fixed_income,allocation_infrastructure,allocation_cash,allocation_alternatives,allocation_credit
count,9890.000000,9890.000000,9890.000000,9890.000000,9890.000000,9890.000000,9890.000000
mean,0.607298,0.057812,0.097820,0.054007,0.061783,0.017128,0.043587
std,0.172722,0.029437,0.132814,0.035556,0.070294,0.024452,0.057732
min,0.098200,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1%,0.229678,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5%,0.310000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.475000,0.040000,0.000000,0.030000,0.020000,0.000000,0.000000
50%,0.610000,0.059700,0.035600,0.050000,0.048000,0.000000,0.028500
75%,0.760000,0.076000,0.158375,0.078000,0.080000,0.030000,0.060000
95%,0.870000,0.100000,0.375000,0.116900,0.183550,0.066775,0.180000


all seven allocation columns are missing for exactly the same 3,829 rows. That strongly indicates a common cause: those modelling rows simply have no SAA record available, rather than individual asset classes being selectively missing. To be safe, create a missingness flag on one of them.

In [36]:
hist_model_mysuper_features[
    "saa_missing"
] = (
    hist_model_mysuper_features[
        "allocation_equity"
    ].isna()
)

# check result
hist_model_mysuper_features[
    "saa_missing"
].value_counts(dropna=False)

saa_missing
False    9890
True     3829
Name: count, dtype: int64

In [37]:
# inspect the allocation fixed income excluding credit
hist_model_mysuper_features[
    "allocation_fixed_income_excluding_credit"
].describe(
    percentiles=[
        0.01, 0.05, 0.25,
        0.50, 0.75, 0.95, 0.99
    ]
)

count    9890.000000
mean        0.049395
std         0.085436
min         0.000000
1%          0.000000
5%          0.000000
25%         0.000000
50%         0.000000
75%         0.070000
95%         0.220000
99%         0.394977
max         0.555200
Name: allocation_fixed_income_excluding_credit, dtype: float64

In [38]:
hist_model_mysuper_features[
    "allocation_fixed_income_excluding_credit"
].isna().sum()

np.int64(3829)

In [39]:
# define first baseline SAA features:
saa_core_features = [
    "strategic_growth_allocation",
    "allocation_equity",
    "allocation_property",
    "allocation_infrastructure",
    "allocation_cash",
    "allocation_alternatives",
    "allocation_credit",
    "allocation_fixed_income_excluding_credit",
    "saa_missing",
]

#### Currency Hedging

In [40]:
# define hedging features
hedging_features = [
    "weighted_currency_hedging_ratio",
    "currency_hedging_applicable_allocation",
    "currency_hedging_distinct_ratios",
    "has_applicable_hedging_allocation",
    "currency_hedging_fully_unhedged",
    "currency_hedging_fully_hedged",
    "currency_hedging_partial",
]

# inspect missingness
hist_model_mysuper_features[
    hedging_features
].isna().sum()

weighted_currency_hedging_ratio           3829
currency_hedging_applicable_allocation    3829
currency_hedging_distinct_ratios          3829
has_applicable_hedging_allocation         3829
currency_hedging_fully_unhedged           3829
currency_hedging_fully_hedged             3829
currency_hedging_partial                  3829
dtype: int64

In [41]:
# inspect distribution
hist_model_mysuper_features[
    [
        "weighted_currency_hedging_ratio",
        "currency_hedging_applicable_allocation",
        "currency_hedging_distinct_ratios",
    ]
].describe(
    percentiles=[
        0.01, 0.05, 0.25,
        0.50, 0.75, 0.95, 0.99
    ]
)

,weighted_currency_hedging_ratio,currency_hedging_applicable_allocation,currency_hedging_distinct_ratios
count,9890.000000,9890.000000,9890.0
mean,0.501834,0.539616,2.814358
std,0.195682,0.077683,0.996868
min,0.000000,0.166400,1.0
1%,0.000000,0.310000,1.0
5%,0.000000,0.392065,1.0
25%,0.402584,0.503425,2.0
50%,0.520990,0.550000,3.0
75%,0.625002,0.584000,3.0
95%,0.778676,0.653000,5.0


In [42]:
# inspect categorical features
for col in [
    "has_applicable_hedging_allocation",
    "currency_hedging_fully_unhedged",
    "currency_hedging_fully_hedged",
    "currency_hedging_partial",
]:
    print(col + ":")
    print(
        hist_model_mysuper_features[
            col
        ].value_counts(dropna=False)
    )
    print()

has_applicable_hedging_allocation:
has_applicable_hedging_allocation
True    9890
<NA>    3829
Name: count, dtype: Int64

currency_hedging_fully_unhedged:
currency_hedging_fully_unhedged
False    9226
None     3829
True      664
Name: count, dtype: int64

currency_hedging_fully_hedged:
currency_hedging_fully_hedged
False    9773
None     3829
True      117
Name: count, dtype: int64

currency_hedging_partial:
currency_hedging_partial
True     9109
None     3829
False     781
Name: count, dtype: int64



the 3,829 missing hedging rows are the same SAA-unavailable rows. Therefore its missingness can be inferred to the SAA missingness flag. 

Based on the median of the numerical features, a typical observation has roughly half of its applicable exposure hedged, and hedging treatment can vary across several underlying allocation components.

The boolean counts also form a complete partition: 664 unhedged + 117 fully hedged + 9109 partially hedged = 9890 hedging allocation. This means that every SAA-available row belongs to exactly one of those three categories.

Those three categories are largely another representation of the hedging-ratio state. Including the continuous ratio plus all three categories adds redundancy without giving the first model much new information. Because of that, for the first baseline model, the hedging features are not including those 3 categories.

In [43]:
hedging_core_features = [
    "weighted_currency_hedging_ratio",
    "currency_hedging_applicable_allocation",
    "currency_hedging_distinct_ratios",
]

#### Changes in investment strategy over time

This feature basically ask: Compared with the immediately previous quarter, has this investment option become more growth-oriented or more defensive?

In [44]:
# create the previous-quarter strategic growth value
hist_model_mysuper_features[
    "strategic_growth_lag_1q"
] = (
    hist_model_mysuper_features
    .groupby(entity_keys)[
        "strategic_growth_allocation"
    ]
    .shift(1)
)

# calculate the change
hist_model_mysuper_features[
    "strategic_growth_change_1q"
] = (
    hist_model_mysuper_features[
        "strategic_growth_allocation"
    ]
    -
    hist_model_mysuper_features[
        "strategic_growth_lag_1q"
    ]
)

In [45]:
hist_model_mysuper_features[
    "strategic_growth_change_1q"
].isna().sum()

np.int64(4189)

In [46]:
hist_model_mysuper_features[
    "strategic_growth_change_1q"
].describe(
    percentiles=[
        0.01, 0.05, 0.25,
        0.50, 0.75, 0.95, 0.99
    ]
)

count    9.530000e+03
mean     1.556582e-03
std      1.531843e-02
min     -9.875000e-02
1%      -3.000000e-02
5%      -1.500000e-02
25%     -1.110223e-16
50%      0.000000e+00
75%      2.500000e-04
95%      1.925000e-02
99%      7.860000e-02
max      1.800000e-01
Name: strategic_growth_change_1q, dtype: float64

strategic growth allocation is usually very stable from quarter to quarter. most options barely change their strategic growth positioning, while a smaller group makes material changes.

Missingness makes sense because there are 3829 rows where SAA itself is unavailable, and this change feature additionally requires the previous quarter's strategic-growth value. So a row can have current strategic growth available but still have no change value if it is the first observation or the previous quarter's SAA was unavailable.

Cash and equity strategy changes could be created for the baseline model because equity captures changes in major growth exposure and cash can capture shifts toward or away from defensive/liquid positioning.

In [47]:
# create cash and equity strategy changes
for col, feature_name in [
    ("allocation_equity", "equity_allocation_change_1q"),
    ("allocation_cash", "cash_allocation_change_1q"),
]:
    lagged = (
        hist_model_mysuper_features
        .groupby(entity_keys)[col]
        .shift(1)
    )

    hist_model_mysuper_features[
        feature_name
    ] = (
        hist_model_mysuper_features[col]
        - lagged
    )

# inspect missingness
hist_model_mysuper_features[
    [
        "equity_allocation_change_1q",
        "cash_allocation_change_1q",
    ]
].isna().sum()

equity_allocation_change_1q    4189
cash_allocation_change_1q      4189
dtype: int64

In [48]:
hist_model_mysuper_features[
    [
        "equity_allocation_change_1q",
        "cash_allocation_change_1q",
    ]
].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

,equity_allocation_change_1q,cash_allocation_change_1q
count,9530.000000,9530.000000
mean,0.002574,0.000236
std,0.020489,0.014338
min,-0.123000,-0.321900
1%,-0.035000,-0.038213
5%,-0.015000,-0.011000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
95%,0.026325,0.015000


The missingness result shows that both strategies basically have the same missingness as the strategic growth allocation.

Equity allocation changes are less stable compared to strategic allocation and cash allocation but at least half of the observations have essentially no quarter-to-quarter change in these allocations. That makes sense for strategic asset allocation, which generally should not be moving dramatically every quarter.

The tails show substantial shifts, especially the -0.3219 cash movement. That cash minimum is something worth retaining as a known extreme observation if it later strongly influences the model.

In [49]:
# save the latest dataset into a parquet file to be used for modelling
hist_model_mysuper_features.to_parquet(
    processed_dir / "hist_model_mysuper_features.parquet",
    index=False,
)